In [1]:
import cv2
from pathlib import Path
from tqdm import tqdm

input_dir = Path("segmented/mask")
output_dir = Path("segmented/mask_vert")
output_dir.mkdir(parents=True, exist_ok=True)

image_paths = list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))

print(f"Standardizing orientation for {len(image_paths)} images...")

skipped = 0
processed = 0
failed = 0

for img_path in tqdm(image_paths):
    out_path = output_dir / img_path.name

    # Skip if already exists
    if out_path.exists():
        skipped += 1
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        failed += 1
        continue
        
    h, w = img.shape[:2]
    
    # If horizontal (width > height), rotate 90 degrees
    if w > h:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    
    success = cv2.imwrite(str(out_path), img)
    if success:
        processed += 1
    else:
        failed += 1

print(f"Done!")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")
print(f"Failed: {failed}")
print(f"Output directory: {output_dir}")

Standardizing orientation for 4632 images...


100%|██████████| 4632/4632 [00:00<00:00, 46219.44it/s]

Done!
Processed: 0
Skipped: 4632
Failed: 0
Output directory: segmented\mask_vert


In [2]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_dir = Path("segmented/mask_vert")
output_dir = Path("segmented/mask_deskewed")
output_dir.mkdir(parents=True, exist_ok=True)

image_paths = list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))
print(f"Found {len(image_paths)} images")

failed = []
skipped = 0
processed = 0

for img_path in tqdm(image_paths):
    out_path = output_dir / img_path.name

    # Skip if already exists
    if out_path.exists():
        skipped += 1
        continue

    try:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            failed.append((img_path.name, "Could not read"))
            continue

        _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            failed.append((img_path.name, "No contours found"))
            continue

        largest = max(contours, key=cv2.contourArea)

        if len(largest) < 5:
            failed.append((img_path.name, "Contour too small to fit ellipse"))
            continue

        ellipse = cv2.fitEllipse(largest)
        center, axes, angle = ellipse

        h, w = img.shape
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=0)
        rotated = cv2.rotate(rotated, cv2.ROTATE_180)

        success = cv2.imwrite(str(out_path), rotated)
        if success:
            processed += 1
        else:
            failed.append((img_path.name, "Write failed"))

    except Exception as e:
        failed.append((img_path.name, str(e)))

print("\nDone!")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")
print(f"Failed: {len(failed)}")
print(f"Total: {len(image_paths)}")

if failed:
    print("\nFailed details:")
    for name, reason in failed:
        print(f"  {name}: {reason}")

Found 4632 images


100%|██████████| 4632/4632 [00:00<00:00, 42345.84it/s]


Done!
Processed: 0
Skipped: 4632
Failed: 0
Total: 4632


In [4]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_dir = Path("segmented/mask_deskewed")
output_dir = Path("cleaned_dataset/mask_rotated")
output_dir.mkdir(parents=True, exist_ok=True)

image_paths = list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))
print(f"Found {len(image_paths)} masks for pointy-end-up alignment")

skipped = 0
processed = 0
failed = []

for img_path in tqdm(image_paths):
    out_path = output_dir / img_path.name

    if out_path.exists():
        skipped += 1
        continue

    try:
        mask = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            failed.append((img_path.name, "Could not read"))
            continue

        _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            failed.append((img_path.name, "No contours found"))
            continue
            
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h_box = cv2.boundingRect(cnt)
        
        egg_crop = binary[y:y+h_box, x:x+w]
        
        start_col = int(w * 0.25)
        end_col = int(w * 0.75)
        
        y_top = []
        y_bottom = []
        
        for c in range(start_col, end_col):
            col_data = egg_crop[:, c]
            non_zeros = np.where(col_data > 0)[0]
            if len(non_zeros) > 0:
                y_top.append(non_zeros[0])
                y_bottom.append(non_zeros[-1])
            else:
                y_top.append(0)
                y_bottom.append(0)
                
        if len(y_top) > 5:
            x_vals = np.arange(len(y_top))
            
            p_top = np.polyfit(x_vals, y_top, 2)
            p_bottom = np.polyfit(x_vals, y_bottom, 2)
            
            curve_top = abs(p_top[0])
            curve_bottom = abs(p_bottom[0])
            
            if curve_top < curve_bottom:
                mask = cv2.rotate(mask, cv2.ROTATE_180)
                
        success = cv2.imwrite(str(out_path), mask)
        if success:
            processed += 1
        else:
            failed.append((img_path.name, "Write failed"))
            
    except Exception as e:
        failed.append((img_path.name, str(e)))

print("\nDone!")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")
print(f"Failed: {len(failed)}")
print(f"Total: {len(image_paths)}")

if failed:
    print("\nFailed details:")
    for name, reason in failed:
        print(f"  {name}: {reason}")

Found 4632 masks for pointy-end-up alignment


100%|██████████| 4632/4632 [00:00<00:00, 34459.18it/s]


Done!
Processed: 43
Skipped: 4589
Failed: 0
Total: 4632
